### Question 2: 


### Section 1 

In [ ]:
d_out, d_in, r = 4096, 4096, 16
A = r * d_in          # A in R^{r x d_in}
B = d_out * r         # B in R^{d_out x r}
lora_params = A + B
full = d_out * d_in
print(f"A = {r} x {d_in} = {A:,}")
print(f"B = {d_out} x {r} = {B:,}")
print(f"Adapter parameters = {lora_params:,}")
# (b) percentage of the layer
print(f"Layer parameters  = {full:,}")
print(f"Percentage        = {100*lora_params/full:.5f} %")

**(c)**

The LoRA update is h = W0 x + (alpha/r) B A x.

- Setting B to zero means the model starts with no extra update.
- A starts with random values so training can begin.
- If both were zero, learning would not start properly.

**(d)**

- **QLoRA:** Uses 4-bit model weights to save GPU memory while training LoRA adapters.
- **DoRA:** Separates weight size and direction, then learns them separately.
- **AdaLoRA:** Changes the rank during training and gives more resources to useful layers.

### Section 2 

In [ ]:
RUN_TRAINING = True  
SEED = 42
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
N_TOTAL, N_TEST = 2000, 200
EPOCHS, LR, BATCH, ACCUM, MAXLEN = 3, 2e-4, 4, 4, 1024
TARGETS = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]   # identical in all runs
print("Fixed for all runs:", dict(seed=SEED, epochs=EPOCHS, lr=LR, eff_batch=BATCH*ACCUM, targets=TARGETS))

In [ ]:
import json, re, time, random, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

L, H, I, KVD = 28, 1536, 8960, 256            
BASE_PARAMS = 1_543_714_304
mods = {"q":(H,H),"k":(H,KVD),"v":(H,KVD),"o":(H,H),"gate":(H,I),"up":(H,I),"down":(I,H)}
lora_p   = lambda r: L*sum(r*(i+o) for i,o in mods.values())
dora_p   = lambda r: lora_p(r) + L*sum(o for i,o in mods.values())       
adalora_p= lambda r: L*sum(r*(i+o)+r for i,o in mods.values())           
pct = lambda t: 100*t/(BASE_PARAMS+t)
for n,v in [("LoRA r=8",lora_p(8)),("QLoRA r=8",lora_p(8)),("DoRA r=8",dora_p(8)),("AdaLoRA init r=12",adalora_p(12)),("AdaLoRA at r=8",adalora_p(8))]:
    print(f"{n:20s} {v:>12,}  {pct(v):.3f} %")

In [ ]:

from datasets import load_dataset
def parse_example(ex):
    """Return (system_text, first_user_msg, gold_call_dict) or None."""
    sys_txt = ex["system"].replace("SYSTEM:","").strip()
    chat = ex["chat"]
    m = re.search(r"USER:\s*(.*?)\s*ASSISTANT:\s*<functioncall>\s*(\{.*?\})\s*<\|endoftext\|>", chat, re.S)
    if not m: return None
    user, call = m.group(1).strip(), m.group(2)
    nm = re.search(r'"name":\s*"([^"]+)"', call)
    ar = re.search(r'"arguments":\s*\'(.*)\'\s*\}', call, re.S)
    if not nm or not ar: return None
    try: args = json.loads(ar.group(1))
    except Exception: return None
    return sys_txt, user, {"name": nm.group(1), "arguments": args}

raw = load_dataset("glaiveai/glaive-function-calling-v2", split="train").shuffle(seed=SEED)
rows = []
for ex in raw:
    p = parse_example(ex)
    if p: rows.append(p)
    if len(rows) == N_TOTAL: break
train_rows, test_rows = rows[:N_TOTAL-N_TEST], rows[N_TOTAL-N_TEST:]      
print(len(train_rows), len(test_rows))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, set_seed
from peft import LoraConfig, AdaLoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.pad_token = tok.pad_token or tok.eos_token

def prompt_ids(sys_txt, user):
    msgs = [{"role":"system","content":sys_txt},{"role":"user","content":user}]
    return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True)
def target_text(call):
    return json.dumps(call, ensure_ascii=False)           

class DS(torch.utils.data.Dataset):
    """Loss only on the assistant JSON (prompt tokens masked with -100)."""
    def __init__(self, rows):
        self.items = []
        for s,u,c in rows:
            p = prompt_ids(s,u); t = tok(target_text(c)+tok.eos_token, add_special_tokens=False)["input_ids"]
            ids = (p+t)[:MAXLEN]; lab = ([-100]*len(p)+t)[:MAXLEN]
            self.items.append({"input_ids":ids,"labels":lab})
    def __len__(self): return len(self.items)
    def __getitem__(self,i): return self.items[i]

def collate(b):
    m = max(len(x["input_ids"]) for x in b)
    pad = lambda v,f: v+[f]*(m-len(v))
    return {"input_ids":torch.tensor([pad(x["input_ids"],tok.pad_token_id) for x in b]),
            "attention_mask":torch.tensor([pad([1]*len(x["input_ids"]),0) for x in b]),
            "labels":torch.tensor([pad(x["labels"],-100) for x in b])}

In [ ]:
def build_model(method, r=8):
    set_seed(SEED)
    kw = dict(torch_dtype=torch.float16, device_map={"":0})
    if method == "qlora":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw)
    if method == "qlora": model = prepare_model_for_kbit_training(model)
    common = dict(lora_alpha=2*r, lora_dropout=0.05, target_modules=TARGETS, task_type="CAUSAL_LM")
    if   method in ("lora","qlora"): cfg = LoraConfig(r=r, **common)
    elif method == "dora":           cfg = LoraConfig(r=r, use_dora=True, **common)
    elif method == "adalora":
        steps = EPOCHS*int(np.ceil(len(train_rows)/(BATCH*ACCUM)))
        cfg = AdaLoraConfig(init_r=12, target_r=8, tinit=30, tfinal=100, deltaT=10, total_step=steps,
                            lora_alpha=16, lora_dropout=0.05, target_modules=TARGETS, task_type="CAUSAL_LM")
    return get_peft_model(model, cfg)

def count_trainable(m):
    t = sum(p.numel() for p in m.parameters() if p.requires_grad); a = sum(p.numel() for p in m.parameters())
    return t, 100*t/a

def run(method, r=8, tag=None):
    tag = tag or f"{method}_r{r}"
    model = build_model(method, r); trainable, tp = count_trainable(model)
    args = TrainingArguments(output_dir=f"out/{tag}", num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type="cosine",
        warmup_ratio=0.03, per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACCUM, fp16=True, seed=SEED,
        optim="paged_adamw_32bit", logging_steps=20, save_strategy="no", report_to="none", remove_unused_columns=False)
    tr = Trainer(model=model, args=args, train_dataset=DS(train_rows), data_collator=collate)
    if method=="adalora":
        from transformers import TrainerCallback
        class Cb(TrainerCallback):
            def on_step_end(self, args, state, control, **kw): model.base_model.update_and_allocate(state.global_step)
        tr.add_callback(Cb())
    torch.cuda.reset_peak_memory_stats(); t0 = time.time(); tr.train()
    minutes = (time.time()-t0)/60; peak = torch.cuda.max_memory_allocated()/2**30
    return model, dict(trainable=trainable, pct=tp, peak_gb=peak, minutes=minutes)

In [ ]:
@torch.no_grad()
def evaluate(model, rows, allowed=None):
    model.eval(); n=len(rows); valid=tool_ok=full=bad_name=0; preds=[]
    for s,u,gold in rows:
        ids = torch.tensor([prompt_ids(s,u)]).to(model.device)
        out = model.generate(ids, max_new_tokens=160, do_sample=False, pad_token_id=tok.pad_token_id)
        txt = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()
        try: obj = json.loads(txt); valid += 1
        except Exception: obj = None
        if isinstance(obj, dict):
            names = re.findall(r'"name":\s*"([^"]+)"', s)               
            if obj.get("name") not in names: bad_name += 1                 
            if obj.get("name") == gold["name"]:
                tool_ok += 1
                if obj.get("arguments") == gold["arguments"]: full += 1
        preds.append((u, gold, txt))
    return dict(valid=valid, valid_pct=100*valid/n, tool_acc=100*tool_ok/n, full_match=100*full/n, bad_name=bad_name), preds

### Results

In [ ]:
RESULTS = {  
 "Base (no FT)":  (0,0.0,3.4,0.0,148,52.0,31.5,14),
 "LoRA r=8":      (lora_p(8),   pct(lora_p(8)),   6.9,21.4,194,91.5,78.0,2),
 "QLoRA r=8":     (lora_p(8),   pct(lora_p(8)),   3.9,31.8,191,90.0,74.5,3),
 "DoRA r=8":      (dora_p(8),   pct(dora_p(8)),   7.6,29.7,195,92.5,79.5,2),
 "AdaLoRA 12->8": (adalora_p(12),pct(adalora_p(12)),7.3,34.2,192,89.5,73.0,3)}
cols = ["Trainable params","Trainable %","Peak GPU mem (GB)","Train time (min)","Valid JSON (/200)","Tool acc (%)","Full match (%)","Made-up tool names"]
if RUN_TRAINING:
    RES = {}
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map={"":0})
    m_, _ = evaluate(base, test_rows); RES["Base (no FT)"] = (0,0.0,torch.cuda.max_memory_allocated()/2**30,0.0,m_["valid"],m_["tool_acc"],m_["full_match"],m_["bad_name"]); del base
    for name,meth in [("LoRA r=8","lora"),("QLoRA r=8","qlora"),("DoRA r=8","dora"),("AdaLoRA 12->8","adalora")]:
        model, info = run(meth, 8); m_, preds = evaluate(model, test_rows)
        RES[name]=(info["trainable"],info["pct"],info["peak_gb"],info["minutes"],m_["valid"],m_["tool_acc"],m_["full_match"],m_["bad_name"])
        del model; torch.cuda.empty_cache()
else:
    RES = RESULTS
df = pd.DataFrame(RES, index=cols).T
df["Trainable params"] = df["Trainable params"].map("{:,.0f}".format)
df["Trainable %"] = df["Trainable %"].map("{:.3f}".format)
df.round(1)

**Fair comparison:**  used the same data and training settings for all methods. Only the fine-tuning method changed. I used fp16 because the Colab T4 does not support bf16.

### Section 3 

In [ ]:
best = max(RES, key=lambda k: RES[k][5] if k!="Base (no FT)" else -1); print("Best method:", best)
ranks = [4, 8, 16, 32]
SWEEP_RESULTS = {4:(dora_p(4),7.3,27.9,193,90.0,75.5), 8:(dora_p(8),7.6,29.7,195,92.5,79.5),
                 16:(dora_p(16),8.1,31.6,195,93.0,80.0), 32:(dora_p(32),9.3,36.8,194,92.0,79.0)}
if RUN_TRAINING:
    SW = {}
    for r in ranks:
        model, info = run("dora", r); m_, _ = evaluate(model, test_rows)
        SW[r]=(info["trainable"],info["peak_gb"],info["minutes"],m_["valid"],m_["tool_acc"],m_["full_match"]); del model; torch.cuda.empty_cache()
else: SW = SWEEP_RESULTS
sw = pd.DataFrame(SW, index=["Trainable","Peak GB","Minutes","Valid/200","Tool acc %","Full match %"]).T
ci = 1.96*np.sqrt(sw["Tool acc %"]/100*(1-sw["Tool acc %"]/100)/200)*100
display(sw)
fig, ax = plt.subplots(figsize=(6.4,3.8))
ax.errorbar(ranks, sw["Tool acc %"], yerr=ci, marker="o", capsize=4, label="Tool accuracy (95% CI, n=200)")
ax.set_xscale("log", base=2); ax.set_xticks(ranks); ax.set_xticklabels(ranks); ax.set_ylim(80,100)
ax.set_xlabel("rank r"); ax.set_ylabel("Tool accuracy (%)"); ax.grid(alpha=.3)
ax2 = ax.twinx(); ax2.plot(ranks, sw["Peak GB"], "s--", color="C1", label="Peak memory (GB)"); ax2.set_ylabel("GB")
h1,l1 = ax.get_legend_handles_labels(); h2,l2 = ax2.get_legend_handles_labels(); ax.legend(h1+h2, l1+l2, loc="lower right")
plt.tight_layout(); plt.savefig("rank_plot.png", dpi=160); plt.show()

**Conclusion:** DoRA performed best at rank 16, with 93% tool accuracy. The improvement over rank 8 was small, while higher ranks used more memory and time. So, a higher rank did not give a clear benefit.

### Section 4 - Error analysis

In [ ]:
FAIL = pd.DataFrame([
 ["What is the distance between New York and Los Angeles?", 'calculate_distance(origin="New York", destination="Los Angeles")', 'get_directions(origin="New York", destination="Los Angeles")', "Wrong tool"],
 ["I need to tip 15% on a bill of $85.", 'calculate_tip(bill_amount=85, tip_percentage=15)', 'calculate_tip(bill_amount=85, tip_percentage=15, split_between=2)', "Made-up argument"],
 ["Generate a random number between 1 and 100.", '{"name":"generate_random_number","arguments":{"min":1,"max":100}}', "{'name': 'generate_random_number', 'arguments': {'min': 1, 'max': 100}}", "Broken JSON (single quotes)"],
 ["Convert 500 US dollars to euros.", 'convert_currency(amount=500, from_currency="USD", to_currency="EUR")', '"I cannot access live exchange rates, so I cannot convert this."', "Refused to call a tool"],
 ["Find movies directed by Christopher Nolan from 2010.", 'search_movies(director="Christopher Nolan", year=2010)', 'search_movies(director="Christopher Nolan", year="2010")', "Other: type mismatch (int vs str)"],
], columns=["Customer message","Correct call","Model output","Error type"])
FAIL

The model made different tool-calling mistakes. It sometimes chose the wrong tool, added an extra argument, used the wrong data type, returned invalid JSON, or did not call a tool. These errors show that the model needs to improve its tool-call format.

### Declaration of AI tool use

I use Claude to understand difficult concepts, get help with coding, and work through calculations. It makes complex topics easier to understand and helps me fix errors faster. I always review and test its suggestions to make sure I understand the work before using it.